# PeakCut Studio — Kaggle worker

Run the real PeakCut application behind your permanent Cloudflare dashboard.
**First deploy the dashboard using the setup guide.** Select **GPU T4 ×2**, enable Internet, and keep this notebook private.

Add and enable these Kaggle Secrets under Add-ons → Secrets:
- `PEAKCUT_OWNER_EMAIL`: your PeakCut login email.
- `PEAKCUT_OWNER_PASSWORD`: a password of at least 12 characters.
- `PEAKCUT_BRIDGE_TOKEN`: the same random 32+ character secret configured as `BRIDGE_TOKEN` in Cloudflare.

The notebook does not train a new model. It runs pinned PeakCut code with local speech recognition and optional Ollama.
GPU speech recognition is probed; if unavailable it falls back to CPU. English and Arabic are the upstream languages.
Setup downloads several GB on the first run. No paid AI API is required. Kaggle quotas and runtime limits still apply.


## 1. Configure
Paste your permanent dashboard origin below. The dashboard is at `/control`; the full PeakCut editor is at `/app`.
Start with one short, uploaded English video to verify the pipeline before trying YouTube or live streams.


In [ ]:
DASHBOARD_URL = "https://YOUR-WORKER.YOUR-SUBDOMAIN.workers.dev"
SESSION_MINUTES = 120
USE_GPU_ASR = True
USE_LOCAL_AI = True
# Optional: attach your prior encrypted backup as a PRIVATE Kaggle dataset.
RESTORE_BACKUP = ""  # e.g. /kaggle/input/my-private-backup/peakcut-state.enc


## 2. Load the launcher
This writes no credentials into the notebook or saved output.

In [ ]:
ADAPTERS = {'owner_app.py': '"""Notebook adapter: preserve upstream auth, provision one owner, close public signup."""\nimport os\nimport sqlalchemy as sa\nfrom fastapi.responses import JSONResponse\nfrom studio import auth, config, db\nfrom studio.server import app\n\n\ndef provision():\n    db.init()\n    email = os.environ[\'PEAKCUT_OWNER_EMAIL\'].strip().lower()\n    password = os.environ[\'PEAKCUT_OWNER_PASSWORD\']\n    if len(password) < 12:\n        raise ValueError(\'Owner password must have at least 12 characters\')\n    with db.engine.connect() as conn:\n        user = conn.execute(sa.select(db.users).where(db.users.c.email == email)).first()\n    if user is None:\n        user, _ = auth.signup(email, password, \'Creator\')\n    else:\n        auth.login(email, password)  # never silently overwrite a restored password\n    with db.engine.begin() as conn:\n        conn.execute(db.users.update().where(db.users.c.id == user.id).values(is_admin=True))\n    auth.mark_verified(user.id)\n    return user.id\n\n\n@app.middleware(\'http\')\nasync def private_install(request, call_next):\n    # Social OAuth remains enabled; Google website sign-in could create extra users.\n    if request.url.path in (\'/signup\', \'/api/auth/signup\') or request.url.path.startswith(\'/auth/google/\'):\n        return JSONResponse({\'detail\': \'Private installation. Sign in with the notebook owner account.\'}, status_code=403)\n    return await call_next(request)\n\n\nif __name__ == \'__main__\':\n    import uvicorn\n    provision()\n    uvicorn.run(app, host=\'127.0.0.1\', port=config.PORT, proxy_headers=False, log_level=\'warning\')\n', 'bridge.py': '"""Send allowlisted progress, never social tokens, to the permanent dashboard."""\nimport os\nimport time\nimport requests\n\n\ndef sanitize_jobs(rows):\n    result = []\n    for row in rows[:100]:\n        item = {k: row.get(k) for k in (\'id\', \'title\', \'status\', \'stage\', \'duration\', \'error\')}\n        item[\'clips\'] = [{k: c.get(k) for k in (\'id\', \'title\', \'status\', \'start\', \'end\')} for c in row.get(\'clips\', [])[:20]]\n        result.append(item)\n    return result\n\n\ndef run(stop, public_url, tunnel_url, run_id):\n    local = requests.Session()\n    # HTTPS cookie is deliberately supplied explicitly for loopback requests only.\n    response = local.post(\'http://127.0.0.1:8765/api/auth/login\', json={\n        \'email\': os.environ[\'PEAKCUT_OWNER_EMAIL\'], \'password\': os.environ[\'PEAKCUT_OWNER_PASSWORD\']},\n        headers={\'Origin\': public_url}, timeout=20)\n    response.raise_for_status()\n    cookie = response.cookies.get(\'peakcut_session\')\n    if not cookie:\n        raise RuntimeError(\'PeakCut login did not return a session cookie\')\n    headers = {\'Cookie\': \'peakcut_session=\' + cookie}\n    errors = 0\n    while not stop.is_set():\n        try:\n            jobs = local.get(\'http://127.0.0.1:8765/api/jobs\', headers=headers, timeout=20)\n            jobs.raise_for_status()\n            accounts = local.get(\'http://127.0.0.1:8765/api/accounts\', headers=headers, timeout=20)\n            accounts.raise_for_status()\n            posting = local.get(\'http://127.0.0.1:8765/api/overview\', headers=headers, timeout=20)\n            # Posting summary is read separately; no account payload is forwarded wholesale.\n            overview = posting.json() if posting.ok else {}\n            payload = {\'run_id\': run_id, \'protocol\': 2, \'tunnel_url\': tunnel_url, \'jobs\': sanitize_jobs(jobs.json()),\n                       \'accounts\': [{k: a.get(k) for k in (\'id\', \'platform\', \'name\')} for a in accounts.json()],\n                       \'posting\': {\'posts\': [{k: p.get(k) for k in (\'id\', \'platform\', \'status\', \'due_at\', \'posted_at\', \'error\')}\n                                             for p in overview.get(\'posts\', [])[:100]],\n                                   \'recording\': overview.get(\'recording\', 0), \'paused\': overview.get(\'paused\', False)}, \'online\': True}\n            # The overview is public upstream state; the receiver also limits its size.\n            out = requests.post(public_url + \'/bridge/heartbeat\', json=payload,\n                headers={\'Authorization\': \'Bearer \' + os.environ[\'PEAKCUT_BRIDGE_TOKEN\']}, timeout=25)\n            out.raise_for_status()\n            if out.json().get(\'stop_requested\'):\n                print(\'Stop requested from the dashboard. Ending the processing session.\', flush=True)\n                stop.set()\n            errors = 0\n        except Exception as exc:\n            errors += 1\n            print(\'Dashboard heartbeat failed:\', type(exc).__name__, \'(retrying)\', flush=True)\n        stop.wait(min(60, 20 + errors * 5))\n    try:\n        requests.post(public_url + \'/bridge/offline\', json={\'run_id\': run_id},\n            headers={\'Authorization\': \'Bearer \' + os.environ[\'PEAKCUT_BRIDGE_TOKEN\']}, timeout=15)\n    except requests.RequestException:\n        pass\n', 'state_io.py': '"""Encrypted settings/history backup. Media are exported separately, not embedded here."""\nimport io\nimport os\nimport sqlite3\nimport tempfile\nimport zipfile\nfrom pathlib import Path\nfrom cryptography.hazmat.primitives import hashes\nfrom cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC\nfrom cryptography.hazmat.primitives.ciphers.aead import AESGCM\n\nMAGIC = b\'PEAKCUT1\'\n\n\ndef key(password, salt):\n    return PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=600000).derive(password.encode())\n\n\ndef backup(root, destination, password):\n    root, destination = Path(root), Path(destination)\n    buffer = io.BytesIO()\n    total = 0\n    with zipfile.ZipFile(buffer, \'w\', zipfile.ZIP_DEFLATED) as archive:\n        for f in root.rglob(\'*.json\'):\n            relative = f.relative_to(root)\n            if \'tmp\' in relative.parts or \'models\' in relative.parts or f.is_symlink():\n                continue\n            total += f.stat().st_size\n            if total > 100 * 1024 * 1024:\n                raise ValueError(\'Metadata backup exceeds 100 MB. Download clips and archive your workspace separately.\')\n            archive.write(f, relative.as_posix())\n        database = root / \'peakcut.db\'\n        if database.exists():\n            with tempfile.TemporaryDirectory() as folder:\n                snapshot = Path(folder) / \'peakcut.db\'\n                with sqlite3.connect(database) as source, sqlite3.connect(snapshot) as target:\n                    source.backup(target)\n                archive.write(snapshot, \'peakcut.db\')\n    salt, nonce = os.urandom(16), os.urandom(12)\n    encrypted = AESGCM(key(password, salt)).encrypt(nonce, buffer.getvalue(), MAGIC)\n    destination.write_bytes(MAGIC + salt + nonce + encrypted)\n\n\ndef restore(source, root, password):\n    source, root = Path(source), Path(root)\n    if (root / \'peakcut.db\').exists():\n        raise ValueError(\'Restore requires a fresh workspace. Do not overwrite a running database.\')\n    payload = source.read_bytes()\n    if not payload.startswith(MAGIC):\n        raise ValueError(\'Not a PeakCut encrypted backup\')\n    plain = AESGCM(key(password, payload[8:24])).decrypt(payload[24:36], payload[36:], MAGIC)\n    with zipfile.ZipFile(io.BytesIO(plain)) as archive:\n        if sum(i.file_size for i in archive.infolist()) > 120 * 1024 * 1024:\n            raise ValueError(\'Backup is too large\')\n        for item in archive.infolist():\n            target = (root / item.filename).resolve()\n            if not target.is_relative_to(root.resolve()) or item.is_dir():\n                raise ValueError(\'Invalid backup path\')\n            if target.suffix != \'.json\' and item.filename != \'peakcut.db\':\n                raise ValueError(\'Unexpected backup file\')\n        for item in archive.infolist():\n            target = root / item.filename\n            target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_bytes(archive.read(item))\n\n\nif __name__ == \'__main__\':\n    import sys\n    password = os.environ[\'PEAKCUT_OWNER_PASSWORD\']\n    if sys.argv[1] == \'backup\':\n        backup(sys.argv[2], sys.argv[3], password)\n    elif sys.argv[1] == \'restore\':\n        restore(sys.argv[2], sys.argv[3], password)\n'}
"""Kaggle launcher. Executed from the generated notebook; no runtime installation on Windows."""
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import time
import uuid
from pathlib import Path
import requests

COMMIT = '39c32a878648619e981b57af608ba4571a82247f'
ROOT = Path('/tmp/peakcut')
ENV = Path('/tmp/peakcut-env')
DATA = Path('/tmp/peakcut-data')
OUTPUT = Path('/kaggle/working/peakcut-exports')
PROCESSES = []
STOP = threading.Event()


def command(args, **kwargs):
    subprocess.run([str(x) for x in args], check=True, **kwargs)


def secret(name, required=True):
    value = os.environ.get(name, '')
    if not value:
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(name)
        except Exception:
            pass
    if required and not value:
        raise RuntimeError(f'Add and enable Kaggle Secret {name}, then rerun. Never paste it in notebook output.')
    return value


def download(url, dest):
    dest = Path(dest)
    part = dest.with_suffix('.part')
    with requests.get(url, stream=True, timeout=(30, 120)) as response:
        response.raise_for_status()
        with part.open('wb') as f:
            for chunk in response.iter_content(1024 * 1024):
                f.write(chunk)
    part.replace(dest)


def launch(args, name, env):
    log = Path('/tmp') / ('peakcut-' + name + '.log')
    with log.open('w') as f:
        process = subprocess.Popen([str(a) for a in args], cwd=ROOT, env=env, stdout=f, stderr=subprocess.STDOUT,
                                   start_new_session=True)
    PROCESSES.append(process)
    return process, log


def wait_http(url, process, seconds=120):
    for _ in range(seconds):
        if process.poll() is not None:
            raise RuntimeError('A service exited. Read the diagnostic log cell.')
        try:
            if requests.get(url, timeout=2).ok:
                return
        except requests.RequestException:
            pass
        time.sleep(1)
    raise TimeoutError('Service startup timed out: ' + url)


def setup(use_gpu_asr=True, restore_backup=''):
    if not Path('/kaggle').exists():
        raise RuntimeError('Run this notebook in Kaggle, not on your Windows computer.')
    if PROCESSES and any(p.poll() is None for p in PROCESSES):
        raise RuntimeError('Stop the previous session before rerunning setup.')
    for key in ('PEAKCUT_OWNER_EMAIL', 'PEAKCUT_OWNER_PASSWORD', 'PEAKCUT_BRIDGE_TOKEN'):
        os.environ[key] = secret(key)
    if len(os.environ['PEAKCUT_OWNER_PASSWORD']) < 12:
        raise ValueError('Set an owner password with at least 12 characters.')
    if len(os.environ['PEAKCUT_BRIDGE_TOKEN']) < 32:
        raise ValueError('Use at least 32 random characters for the bridge token.')
    OUTPUT.mkdir(parents=True, exist_ok=True)
    DATA.mkdir(parents=True, exist_ok=True)
    command(['apt-get', 'update', '-qq'])
    command(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'git', 'zstd'])
    if not ROOT.exists():
        command(['git', 'clone', '--no-checkout', 'https://github.com/haziqahmedqasmi/PeakCut.git', ROOT])
    command(['git', '-C', ROOT, 'checkout', '--detach', COMMIT])
    try:
        command([sys.executable, '-m', 'venv', ENV])
    except subprocess.CalledProcessError:
        print('Built-in venv/ensurepip failed; using virtualenv bootstrap instead.', flush=True)
        bootstrap = Path('/tmp/peakcut-virtualenv.pyz')
        download('https://bootstrap.pypa.io/virtualenv.pyz', bootstrap)
        command([sys.executable, bootstrap, '--python', sys.executable, ENV])
    py = ENV / 'bin/python'
    command([py, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
    command([py, '-m', 'pip', 'install', '-q', '-r', ROOT / 'requirements.txt', 'requests', 'cryptography', 'deno', 'wrapt'])
    (ROOT / 'state_io.py').write_text(ADAPTERS['state_io.py'], encoding='utf-8')
    if restore_backup:
        command([py, ROOT / 'state_io.py', 'restore', restore_backup, DATA], env=dict(os.environ))
        print('Restored encrypted account settings and history. Media must be restored separately if needed.')
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
    names = gpu.stdout.strip().splitlines() if gpu and gpu.returncode == 0 else []
    print('CPU cores:', os.cpu_count(), '| GPUs:', ', '.join(names) or 'none', flush=True)
    env = dict(os.environ, PEAKCUT_DATA=str(DATA), PEAKCUT_MODELS='/tmp/peakcut-models', PEAKCUT_HOST='127.0.0.1',
               PEAKCUT_PORT='8765', PEAKCUT_ACCOUNTS='1', PEAKCUT_UPDATE_CHECK='0', PEAKCUT_FETCH_MODEL='0',
               PEAKCUT_JOB_WORKERS='1', PEAKCUT_MAX_LIVE='1', PEAKCUT_MAX_LIVE_HOURS='2',
               PEAKCUT_MAX_USER_GB='12', PEAKCUT_ASR_THREADS='2', PEAKCUT_ASR_COPIES='1',
               PYTHONUNBUFFERED='1', PATH=str(ENV / 'bin') + ':' + os.environ['PATH'])
    if names and use_gpu_asr:
        print('Installing the upstream CUDA speech-recognition build…', flush=True)
        result = subprocess.run([str(py), '-m', 'pip', 'install', '-q',
            'sherpa-onnx==2.0.1+cuda12.cudnn9.onnxruntime1.28.2', '-f', 'https://k2-fsa.github.io/sherpa/onnx/cuda.html',
            'nvidia-cublas-cu12', 'nvidia-cudnn-cu12>=9,<10', 'nvidia-cufft-cu12', 'nvidia-curand-cu12',
            'nvidia-cuda-runtime-cu12', 'nvidia-cuda-nvrtc-cu12', 'nvidia-nvjitlink-cu12'])
        if result.returncode == 0:
            libraries = list(ENV.glob('lib/python*/site-packages/nvidia/*/lib'))
            env.update(LD_LIBRARY_PATH=':'.join(map(str, libraries)) + ':' + env.get('LD_LIBRARY_PATH', ''),
                       CUDA_VISIBLE_DEVICES='0', PEAKCUT_ASR_PROVIDER='cuda',
                       PEAKCUT_ASR_MODEL='sherpa-onnx-nemo-parakeet-tdt-0.6b-v2-fp16')
        else:
            print('CUDA wheel unavailable for this Python version; restoring CPU speech recognition.', flush=True)
            command([py, '-m', 'pip', 'install', '-q', '--force-reinstall', 'sherpa-onnx==2.0.1'])
    print('Downloading and checking the English speech model. First setup can take several minutes.', flush=True)
    probe = 'from studio import fetch_model,transcribe; import numpy as np; fetch_model.ensure(); transcribe.decode_pieces([np.zeros(16000,dtype=np.float32)]); print("Speech model ready")'
    try:
        command([py, '-c', probe], cwd=ROOT, env=env, timeout=1200)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        if env.get('PEAKCUT_ASR_PROVIDER') != 'cuda':
            raise
        print('CUDA speech probe failed. Retrying the tested upstream CPU path.', flush=True)
        env.update(PEAKCUT_ASR_PROVIDER='cpu', PEAKCUT_ASR_MODEL='sherpa-onnx-nemo-parakeet-tdt-0.6b-v2-int8')
        command([py, '-c', probe], cwd=ROOT, env=env, timeout=1200)
    return py, env, len(names)


def local_ai(py, env, gpu_count, model='qwen2.5:3b'):
    print('Installing Ollama and downloading the local AI model…', flush=True)
    target = Path('/tmp/peakcut-ollama')
    target.mkdir(exist_ok=True)
    binary = target / 'bin/ollama'
    if not binary.exists():
        archive = Path('/tmp/peakcut-ollama.tar.zst')
        download('https://github.com/ollama/ollama/releases/download/v0.40.17/ollama-linux-amd64.tar.zst', archive)
        command(['tar', '--zstd', '-xf', archive, '-C', target])
        archive.unlink()
    ai_env = dict(env, OLLAMA_HOST='127.0.0.1:11434', OLLAMA_MODELS='/tmp/peakcut-llm-models',
                  OLLAMA_NUM_PARALLEL='1', OLLAMA_CONTEXT_LENGTH='8192', OLLAMA_KEEP_ALIVE='10m',
                  CUDA_VISIBLE_DEVICES='1' if gpu_count > 1 else '0')
    process, _ = launch([binary, 'serve'], 'ollama', ai_env)
    wait_http('http://127.0.0.1:11434/api/tags', process)
    command([binary, 'pull', model], env=ai_env, timeout=1800)
    env.update(PEAKCUT_AI_PROVIDER='ollama', PEAKCUT_AI_MODELS=model, PEAKCUT_AI_BASE_URL='http://127.0.0.1:11434/v1')
    # Upstream settings override environment variables. Keep this notebook explicitly local/free.
    (DATA / 'ai.json').write_text(json.dumps({'provider':'ollama','models':[model],
        'base_url':'http://127.0.0.1:11434/v1','key':''}))


def start(py, env, dashboard_url):
    if not re.fullmatch(r'https://[A-Za-z0-9.-]+', dashboard_url):
        raise ValueError('Enter your deployed dashboard HTTPS origin, without a path or trailing slash.')
    env['PEAKCUT_PUBLIC_URL'] = dashboard_url
    for name in ('owner_app.py', 'bridge.py'):
        (ROOT / name).write_text(ADAPTERS[name], encoding='utf-8')
    server, _ = launch([py, ROOT / 'owner_app.py'], 'server', env)
    wait_http('http://127.0.0.1:8765/healthz', server)
    binary = Path('/tmp/peakcut-cloudflared')
    if not binary.exists():
        download('https://github.com/cloudflare/cloudflared/releases/download/2026.11.2/cloudflared-linux-amd64', binary)
        binary.chmod(0o700)
    tunnel, log = launch([binary, 'tunnel', '--no-autoupdate', '--protocol', 'http2', '--url', 'http://127.0.0.1:8765'], 'tunnel', env)
    tunnel_url = None
    for _ in range(90):
        found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log.read_text(errors='replace'))
        if found:
            tunnel_url = found.group(0)
            break
        if tunnel.poll() is not None:
            break
        time.sleep(1)
    if not tunnel_url:
        raise RuntimeError('Tunnel could not start; see diagnostics. Kaggle networking may restrict it.')
    # Do not display the temporary origin. The permanent Worker origin proxies PeakCut.
    sys.path.insert(0, str(ROOT))
    import bridge
    STOP.clear()
    thread = threading.Thread(target=bridge.run, args=(STOP, dashboard_url, tunnel_url, uuid.uuid4().hex), daemon=True)
    thread.start()
    print('Dashboard: ' + dashboard_url + '/control', flush=True)
    print('PeakCut editor: ' + dashboard_url + '/app', flush=True)
    print('Sign in with your PEAKCUT_OWNER_EMAIL and PEAKCUT_OWNER_PASSWORD. Paste the video link in New video.', flush=True)
    return thread


def export_clips():
    count = 0
    for record in DATA.glob('users/**/job.json'):
        try:
            job = json.loads(record.read_text())
        except (OSError, ValueError):
            continue
        for clip in job.get('clips', []):
            if clip.get('status') != 'done' or not clip.get('file'):
                continue
            source = record.parent / 'clips' / Path(clip['file']).name
            if not source.is_file():
                continue
            dest = OUTPUT / source.relative_to(DATA)
            dest.parent.mkdir(parents=True, exist_ok=True)
            if not dest.exists() or source.stat().st_mtime > dest.stat().st_mtime:
                staging = dest.with_suffix('.part')
                shutil.copy2(source, staging)
                staging.replace(dest)
                count += 1
    return count


def shutdown():
    STOP.set()
    time.sleep(2)
    import signal
    for process in reversed(PROCESSES):
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
    export_clips()
    py = ENV / 'bin/python'
    if py.exists() and (ROOT / 'state_io.py').exists() and (DATA / 'peakcut.db').exists():
        command([py, ROOT / 'state_io.py', 'backup', DATA, OUTPUT / 'peakcut-state.enc'])
        print('Encrypted settings backup saved. Restore with the SAME owner password in a later notebook session.')
    print('Services stopped. Save/download /kaggle/working/peakcut-exports before ending the session.')


## 3. Install and check speech recognition
Watch the cell output for installation and model-download progress.

In [ ]:
if "YOUR-" in DASHBOARD_URL:
    raise ValueError("Set DASHBOARD_URL in cell 1 before installing.")
if not 10 <= SESSION_MINUTES <= 240:
    raise ValueError("Choose a session between 10 and 240 minutes.")
py, peakcut_env, gpu_count = setup(USE_GPU_ASR, globals().get("RESTORE_BACKUP", ""))
print("Speech backend:", peakcut_env.get("PEAKCUT_ASR_PROVIDER", "cpu"))


## 4. Local AI for clip selection
The second GPU is assigned to Ollama when present. GPU 0 is assigned to speech recognition.
The small local model is a practical free starting point; it does not guarantee viral clip selection.
If disabled, PeakCut uses its loudness-based fallback with no AI-written titles.


In [ ]:
if USE_LOCAL_AI:
    local_ai(py, peakcut_env, gpu_count)
else:
    peakcut_env.update(PEAKCUT_AI_PROVIDER="ollama", PEAKCUT_AI_MODELS="")
    (DATA / "ai.json").write_text(json.dumps({"provider":"ollama","models":[],"base_url":"http://127.0.0.1:11434/v1","key":""}))
print("Clip-selection setup finished.")


## 5. Open your dashboard
Open the displayed link, enter your Cloudflare dashboard password, then open the full editor.
Sign in there with your **PeakCut owner email/password**. Paste your video link in New video, or upload a file.
Use the permanent dashboard to see progress; use PeakCut for previews, caption options, accounts and publishing.


In [ ]:
bridge_thread = start(py, peakcut_env, DASHBOARD_URL)
from IPython.display import display, HTML
import html
display(HTML('<a target="_blank" rel="noopener" href="'+html.escape(DASHBOARD_URL+'/control', quote=True)+'" style="display:inline-block;background:#6f4ce8;color:white;padding:18px 28px;border-radius:12px;font-size:20px;text-decoration:none">Open PeakCut dashboard</a>'))


## 6. Run the bounded processing session
Leave this cell running while using the dashboard. For a batch run, Save & Run All keeps the requested computation running without this browser tab.
It stops after the configured duration; this is not an unlimited service or an automatic restart system.
Completed clips are copied to `/kaggle/working/peakcut-exports` every 30 seconds.
Stopping abruptly can interrupt an edit or upload. The encrypted settings backup is made on orderly shutdown.


In [ ]:
deadline = time.monotonic() + SESSION_MINUTES * 60
try:
    while time.monotonic() < deadline and not STOP.is_set():
        if not bridge_thread.is_alive() and not STOP.is_set():
            raise RuntimeError("Progress bridge stopped. Check the diagnostics cell.")
        if any(p.poll() is not None for p in PROCESSES):
            raise RuntimeError("A service stopped. Check the diagnostics cell.")
        copied = export_clips()
        if copied:
            print(f"Saved {copied} completed clip(s) to notebook outputs.", flush=True)
        time.sleep(30)
finally:
    shutdown()


## Diagnostics — run when needed
Logs are local to this notebook. Do not share raw logs if you have connected private accounts.


In [ ]:
for name in ("server", "ollama", "tunnel"):
    path = Path("/tmp/peakcut-" + name + ".log")
    if path.exists():
        print("\n" + name.upper())
        text = path.read_text(errors="replace")[-4000:]
        text = re.sub(r"https?://[^\s]+", "[URL]", text)
        print(text)


## Stop manually
Run only when you want to stop. Download the exported MP4s and `peakcut-state.enc` before discarding the session.
The encrypted backup contains accounts/settings/history, **not video files**. Restore it with the same owner password.
Uploaded source files and unfinished renders are not automatically recoverable. Saved Cloudflare history is separate.


In [ ]:
shutdown()